# Ollama RAG Chatbot Codelab
This notebook explains each part of the application.

## 1. Install Packages

Install required libraries.

In [ ]:
from pexpect.replwrap import python
!pip install streamlit langchain langchain-community langchain-text-splitters langchain-ollama chromadb sentence-transformers pypdf

## Note!!
All code should be written in app.py file
- open app.py in your favourite code editor and start hacking away

## 2. Imports

Import libraries.

In [24]:
#Skip this code as it is just for the notebook
import warnings
warnings.filterwarnings('ignore')

In [25]:
import os
import streamlit as st
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaLLM
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate


## 3. Streamlit Setup

Configure app.

In [26]:
st.set_page_config(page_title="Ollama RAG Chatbot", page_icon="", layout="wide")
st.title(" Local Ollama RAG Chatbot")
st.write("Ask questions about your documents using a 100% local LLM pipeline.")

2026-07-07 19:15:10.619 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-07 19:15:10.620 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-07 19:15:10.620 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-07 19:15:10.621 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-07 19:15:10.622 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


## 4. Data Folder

Create folder if it does not exist.

In [27]:
DATA_DIR="data"
if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR)

## 5. Complete build_ollama_rag()

Build the RAG pipeline and Cache the RAG initialization so your PDFs are processed only once.

This Python function sets up a classic Retrieval-Augmented Generation (RAG) pipeline using the LangChain framework. It is tailored to act as an assistant for INEC Nigeria (Independent National Electoral Commission) by extracting info from PDFs and using a local LLM to answer questions.

It looks inside a specified directory (DATA_DIR) and reads all the PDF files found there. (The knowledge base is populated with the PDFs.)

It breaks the massive text from the PDFs into smaller, manageable pieces ("chunks") of 700 characters each.

The chunk_overlap=100 means consecutive chunks share 100 characters of a text. This ensures context isn't accidentally cut in half mid-sentence at the boundary of a chunk.

We then convert those text chunks into mathematical vectors (numbers) using a lightweight HuggingFace model (all-MiniLM-L6-v2). These vectors capture the semantic meaning of the text.

Then we define the persona and boundaries for the AI. It strictly tells the LLM to behave like an INEC assistant and to use only the injected {context} (the chunks fetched by the retriever) to answer the user's {input}.

OllamaLLM(...): This is a wrapper class from LangChain that allows your Python code to communicate with Ollama, an application that lets you run open-source AI models locally on your own computer (instead of calling an external cloud API like OpenAI, Google, or Anthropic).

model="llama3.2": This specifies the exact AI model you want Ollama to run. Llama 3.2 is a lightweight, fast, and highly efficient open-source model released by Meta, making it perfect for running locally on standard laptop or desktop hardware.

temperature=0.3: This setting controls the "creativity" or randomness of the model's responses on a scale typically from 0 to 1.

In [28]:
@st.cache_resource(show_spinner="Indexing your local PDF files...")
def build_ollama_rag():
    loader=PyPDFDirectoryLoader(DATA_DIR)
    docs=loader.load()
    if not docs:
        return None
    splitter=RecursiveCharacterTextSplitter(chunk_size=700,chunk_overlap=100)
    splits=splitter.split_documents(docs)
    embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    vectorstore=Chroma.from_documents(documents=splits,embedding=embeddings)
    retriever=vectorstore.as_retriever(search_kwargs={"k":3})
    system_prompt=("You are a helpful organizational assistant for INEC Nigeria. "
                   "Use the context to answer. Context:\n{context}")
    prompt=ChatPromptTemplate.from_messages([("system",system_prompt),("human","{input}")])
    llm=OllamaLLM(model="llama3.2",temperature=0.3)
    qa_chain=create_stuff_documents_chain(llm,prompt)
    return create_retrieval_chain(retriever,qa_chain)

## 6. Sidebar

Display indexed files.

In [29]:
# --- Sidebar ---
with st.sidebar:
    st.header("Indexed Documents")
    files = [f for f in os.listdir(DATA_DIR) if f.endswith('.pdf')]
    if files:
        st.success(f"Found {len(files)} local PDF file(s).")
        for f in files:
            st.markdown(f"- `{f}`")
    else:
        st.warning("Please add PDF documents into your local `/data` folder to start.")

2026-07-07 19:15:10.673 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-07 19:15:10.673 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-07 19:15:10.674 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-07 19:15:10.674 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-07 19:15:10.674 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-07 19:15:10.675 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-07 19:15:10.675 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-07 19:15:10.675 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

## 7. Chat

Chat interface.

In [30]:
rag_chain=build_ollama_rag()
if "messages" not in st.session_state:
    st.session_state.messages=[]
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])
if rag_chain is None:
    st.info("The application is ready. Drop some PDFs into the `data/` folder and refresh your page to start chatting.")
else:
    if user_query := st.chat_input("Ask a question about your documents..."):
        # Show User Input
        with st.chat_message("user"):
            st.markdown(user_query)
        st.session_state.messages.append({"role": "user", "content": user_query})

        # Generate Local Ollama Output
        with st.chat_message("assistant"):
            with st.spinner("Ollama is processing chunks and thinking..."):
                try:
                    response = rag_chain.invoke({"input": user_query})
                    answer = response["answer"]
                    st.markdown(answer)
                    st.session_state.messages.append({"role": "assistant", "content": answer})
                except Exception as e:
                    st.error(
                        f"Failed to communicate with Ollama: {e}. Check if `ollama run` is running in your terminal.")

2026-07-07 19:15:10.712 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-07 19:15:10.714 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-07 19:15:10.715 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-07 19:15:10.716 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-07 19:15:10.718 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-07 19:15:10.719 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-07 19:15:10.719 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


## Run Your Code
``` python
streamlit run app.py
```

## Exercise
- Add your own PDFs into the `data` folder and ask questions.
- Customize the interface by adding an inec logo or banner at the top
- Change the Titles to something INEC related